In [ ]:
# =====================================================================
# CELLA IMPORTAZIONI E ATTIVAZIONE GPU APPLE SILICON (M1)
# =====================================================================
import os
import glob
import logging
import gc  # <--- IMPORTANTE: serve per svuotare la RAM satura
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output
from keras.models import load_model

# Silenziamo i log di sistema inutili
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
logging.getLogger('tensorflow').setLevel(logging.ERROR)

import tensorflow as tf
tf.get_logger().setLevel('ERROR')
tf.autograph.set_verbosity(0)

# CONTROLLO E ABILITAZIONE GPU MAC M1 (MPS - Metal Performance Shaders)
dispositivi_gpu = tf.config.list_physical_devices('GPU')
if dispositivi_gpu:
    print(f"Ottimo! GPU M1 Rilevata correttamente: {dispositivi_gpu}")
    # Nota: TensorFlow su Mac gestisce automaticamente l'allocazione su MPS
else:
    print("Nessuna GPU rilevata. Se hai un Mac M1, assicurati di aver installato 'tensorflow-metal'")

from keras import layers, models, losses
from keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau

In [1]:
# =====================================================================
# CELLA IMPORTAZIONI LIBRERIE PER MAC
# =====================================================================
import os
import glob
import logging
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output
from keras.models import load_model

# Silenziamo i log inutili del Mac
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'  # Blocca tutto tranne gli errori fatali
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0' 
logging.getLogger('tensorflow').setLevel(logging.ERROR)

# NOTA: Rimosso 'TF_CUDNN_USE_AUTOTUNE' perché sul tuo Mac non serve

import tensorflow as tf

# Altri silenziatori di log 
tf.get_logger().setLevel('ERROR')
tf.autograph.set_verbosity(0)

# =====================================================================
# MODIFICA 1: DISABILITARE LA GPU PER EVITARE I GRADIENTI NaN (ESPLOSIONE DELLA LOSS)
# =====================================================================
# Diciamo a TensorFlow di "nascondere" la GPU M1 (gestita da tensorflow-metal).
tf.config.set_visible_devices([], 'GPU')

# Riga di controllo per essere sicuri al 100% che abbia funzionato
print("Dispositivi di calcolo attivi:", tf.config.get_visible_devices())
# =====================================================================

# Import di Keras (lasciati identici a quelli del tuo collega)
from keras import layers, models, losses
from keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau

Dispositivi di calcolo attivi: [PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU')]


In [3]:
# ==============================================================================
# DATA ENGINE V10 (Approccio Grid/Heatmap stile FOMO)
# ==============================================================================
GRID_W, GRID_H = 24, 40   # Griglia della stanza (40 righe, 24 colonne)
X_MAX, Y_MAX = 4.8, 7.2   # Dimensioni fisiche della stanza in metri

def coords_to_heatmap(people_xy, people_mask, T):
    """Converte le coordinate (X,Y) in una griglia di probabilità (Heatmap)"""
    heatmaps = np.zeros((T, GRID_H, GRID_W, 1), dtype=np.float32)
    for t in range(T):
        for p in range(4):
            if people_mask[t, p]:
                x, y = people_xy[t, p]
                # Calcola l'indice della cella proporzionale alla dimensione
                col = int(np.clip(x / X_MAX * GRID_W, 0, GRID_W - 1))
                row = int(np.clip(y / Y_MAX * GRID_H, 0, GRID_H - 1))
                # Piazziamo la persona nella griglia
                heatmaps[t, row, col, 0] = 1.0
    return heatmaps

def load_and_process_all_files(file_list, alpha=0.20):
    X_all, Y_all = [], []
    print(f"Inizio caricamento ed EMA Decluttering di {len(file_list)} file...")
    
    for i, file_path in enumerate(file_list):
        data = np.load(file_path)
        raw_iq = data['radar_cir_iq']   
        people_xy = data['people_xy']   
        people_mask = data['people_mask'] 
        T = raw_iq.shape[0]             
        
        mag = np.sqrt(raw_iq[..., 0]**2 + raw_iq[..., 1]**2)
        mag_reshaped = mag.reshape(T, 1, 120, 18) 
        
        bg = np.copy(mag_reshaped[0])
        decluttered = np.zeros_like(mag_reshaped)
        
        for t in range(T):
            bg = alpha * mag_reshaped[t] + (1 - alpha) * bg
            decluttered[t] = np.abs(mag_reshaped[t] - bg)
        
        # NUOVO TARGET: Generiamo le Heatmap invece del flatten delle coordinate
        heatmaps = coords_to_heatmap(people_xy, people_mask, T)

        X_all.append(decluttered)
        Y_all.append(heatmaps)
        
        print(f"File {i+1}/{len(file_list)} processato. ({T} frame pre-calcolati)")

    X = np.concatenate(X_all, axis=0).astype(np.float32)
    Y = np.concatenate(Y_all, axis=0).astype(np.float32)
    return X, Y

# ----------------- SPLIT DEI DATI -----------------
val_indices = [23, 20, 0, 13, 9] 
train_indices = [22, 16, 17, 18, 19, 21, 1, 2, 3, 4, 12, 14, 15, 5, 6, 7, 8, 10, 11]

tutti_i_file = glob.glob("dataset/*.npz") # <-- Assicurati che il path sia giusto
train_files = [f for f in tutti_i_file if int(os.path.basename(f).replace("window_", "").replace(".npz", "")) in train_indices]
val_files = [f for f in tutti_i_file if int(os.path.basename(f).replace("window_", "").replace(".npz", "")) in val_indices]

print("\n--- PREPARAZIONE TRAINING SET ---")
X_train, Y_train = load_and_process_all_files(train_files)

print("\n--- PREPARAZIONE VALIDATION SET ---")
X_val, Y_val = load_and_process_all_files(val_files)

print("\n==================================================")
print(f"DATI TOTALI PRONTI IN RAM! Shape target: {Y_train.shape}")
print("==================================================")


--- PREPARAZIONE TRAINING SET ---
Inizio caricamento ed EMA Decluttering di 19 file...
File 1/19 processato. (7500 frame pre-calcolati)
File 2/19 processato. (7500 frame pre-calcolati)
File 3/19 processato. (7500 frame pre-calcolati)
File 4/19 processato. (7500 frame pre-calcolati)
File 5/19 processato. (7500 frame pre-calcolati)
File 6/19 processato. (7500 frame pre-calcolati)
File 7/19 processato. (7500 frame pre-calcolati)
File 8/19 processato. (7500 frame pre-calcolati)
File 9/19 processato. (7500 frame pre-calcolati)
File 10/19 processato. (7500 frame pre-calcolati)
File 11/19 processato. (7500 frame pre-calcolati)
File 12/19 processato. (7500 frame pre-calcolati)
File 13/19 processato. (7500 frame pre-calcolati)
File 14/19 processato. (7500 frame pre-calcolati)
File 15/19 processato. (7500 frame pre-calcolati)
File 16/19 processato. (7500 frame pre-calcolati)
File 17/19 processato. (7500 frame pre-calcolati)
File 18/19 processato. (7500 frame pre-calcolati)
File 19/19 processato

In [4]:
# =====================================================================
# LOSS E METRICHE PER HEATMAP
# =====================================================================

def heatmap_focal_loss(y_true, y_pred):
    """
    Binary Crossentropy pesata: siccome la griglia è vuota per il 99%,
    diamo molto più peso agli errori fatti sulle celle dove c'è una persona.
    """
    y_pred = tf.clip_by_value(y_pred, 1e-7, 1.0 - 1e-7)
    bce = tf.keras.backend.binary_crossentropy(y_true, y_pred)
    
    # Moltiplichiamo il peso per 20 dove c'è effettivamente una persona
    weight = y_true * 20.0 + 1.0 
    
    return tf.reduce_mean(bce * weight)

def grid_accuracy(y_true, y_pred):
    """Metrica semplice che guarda se la probabilità predice bene il background e i target"""
    y_pred_binary = tf.round(y_pred)
    return tf.keras.metrics.binary_accuracy(y_true, y_pred_binary)

In [ ]:
# ==============================================================================
# ARCHITETTURA FOMO-STYLE (Fully Convolutional con Grid Decoder)
# ==============================================================================
def build_fomo_radar_model(n_radars=6, n_antennas=3, n_bins=120):
    input_channels = n_radars * n_antennas
    inputs = layers.Input(shape=(1, n_bins, input_channels), name="radar_input")

    # 1. Feature Extraction (1D travestito da 2D)
    x = layers.Conv2D(32, (1, 3), strides=(1, 2), padding='same', activation='relu')(inputs) # out: (1, 60, 32)
    x = layers.Conv2D(64, (1, 3), strides=(1, 2), padding='same', activation='relu')(x)      # out: (1, 30, 64)
    x = layers.Conv2D(64, (1, 3), strides=(1, 2), padding='same', activation='relu')(x)      # out: (1, 15, 64)
    
    # Abbiamo 1 * 15 * 64 = 960 elementi.
    # 2. Riorganizzazione Spaziale (Senza layer Dense!)
    # Rimodelliamo 960 elementi in una piccola griglia spaziale (10x6) con 16 canali (10*6*16 = 960)
    x = layers.Reshape((10, 6, 16))(x)
    
    # 3. Spatial Decoder (Upsampling)
    # Raddoppiamo le dimensioni fino ad arrivare alla nostra stanza 40x24
    x = layers.Conv2DTranspose(32, (3, 3), strides=(2, 2), padding='same', activation='relu')(x) # out: (20, 12, 32)
    x = layers.Conv2DTranspose(16, (3, 3), strides=(2, 2), padding='same', activation='relu')(x) # out: (40, 24, 16)
    
    # Output Map: probabilità per ogni cella
    output = layers.Conv2D(1, (1, 1), padding='same', activation='sigmoid', name="grid_output")(x) # out: (40, 24, 1)

    return models.Model(inputs=inputs, outputs=output, name="EEAI_FOMO_Net")

# ==============================================================================
# ADDESTRAMENTO
# ==============================================================================
model_fomo = build_fomo_radar_model()

model_fomo.compile(
    optimizer='adam',
    loss=heatmap_focal_loss, 
    metrics=[grid_accuracy]
)

checkpoint = ModelCheckpoint("eeai_fomo_best.keras", monitor="val_loss", save_best_only=True, verbose=1)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6, verbose=1)
early_stop = EarlyStopping(monitor='val_loss', patience=7, restore_best_weights=True, verbose=1)

EPOCHS = 50 

print("\n--- INIZIO ADDESTRAMENTO FOMO-STYLE ---")
history_light = model_fomo.fit(
    X_train, Y_train,
    validation_data=(X_val, Y_val),
    batch_size=32,
    shuffle=True,
    epochs=EPOCHS,
    callbacks=[checkpoint, reduce_lr, early_stop], 
    verbose=1
)
print("--- ADDESTRAMENTO COMPLETATO ---")

In [5]:
def build_fomo_deep_model(n_radars=6, n_antennas=3, n_bins=120):
    input_channels = n_radars * n_antennas
    inputs = layers.Input(shape=(1, n_bins, input_channels), name="radar_input")

    # --- ENCODER: Estrazione Feature Potenziata ---
    # Layer iniziale
    x = layers.Conv2D(64, (1, 5), padding='same', activation='relu')(inputs)
    x = layers.BatchNormalization()(x)
    
    # Blocco Residuo 1: riduce la dimensione 1D
    res1 = layers.Conv2D(128, (1, 3), strides=(1, 2), padding='same')(x)
    x = layers.Conv2D(128, (1, 3), strides=(1, 2), padding='same', activation='relu')(x)
    x = layers.Conv2D(128, (1, 3), padding='same', activation='relu')(x)
    x = layers.Add()([x, res1])
    x = layers.BatchNormalization()(x) # Shape: (1, 30, 128)

    # Blocco Residuo 2: estrazione features profonde
    res2 = layers.Conv2D(256, (1, 3), strides=(1, 2), padding='same')(x)
    x = layers.Conv2D(256, (1, 3), strides=(1, 2), padding='same', activation='relu')(x)
    x = layers.Conv2D(256, (1, 3), padding='same', activation='relu')(x)
    x = layers.Add()([x, res2])
    x = layers.BatchNormalization()(x) # Shape: (1, 15, 256)

    x = layers.Reshape((20, 12, 32))(x) 
    
    # --- DECODER ---
    # Ora siamo a 20x12, ci basta un solo upsampling per arrivare a 40x24
    x = layers.Conv2DTranspose(16, (3, 3), strides=(2, 2), padding='same', activation='relu')(x)
    
    # Output finale: 40x24x1
    output = layers.Conv2D(1, (3, 3), padding='same', activation='sigmoid', name="grid_output")(x)

    return models.Model(inputs=inputs, outputs=output, name="EEAI_FOMO_Deep")

In [6]:
# Istanza
model_fomo = build_fomo_deep_model()

# Compilazione con un Learning Rate più conservativo
model_fomo.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0005),
    loss=heatmap_focal_loss, 
    metrics=[grid_accuracy]
)

# Callback: diamo più "spazio" al modello per convergere
checkpoint = ModelCheckpoint("eeai_fomo_deep_best.keras", monitor="val_loss", save_best_only=True, verbose=1)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-7, verbose=1)
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1)

print("\n--- INIZIO ADDESTRAMENTO FOMO DEEP ---")
history_deep = model_fomo.fit(
    X_train, Y_train,
    validation_data=(X_val, Y_val),
    batch_size=32,
    shuffle=True,
    epochs=30, # Più epoche per un modello più capace
    callbacks=[checkpoint, reduce_lr, early_stop], 
    verbose=1
)


--- INIZIO ADDESTRAMENTO FOMO DEEP ---
Epoch 1/30
4452/4454 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step - grid_accuracy: 0.9938 - loss: 0.1727
Epoch 1: val_loss improved from None to 0.12843, saving model to eeai_fomo_deep_best.keras

Epoch 1: finished saving model to eeai_fomo_deep_best.keras
4454/4454 ━━━━━━━━━━━━━━━━━━━━ 169s 38ms/step - grid_accuracy: 0.9936 - loss: 0.1454 - val_grid_accuracy: 0.9927 - val_loss: 0.1284 - learning_rate: 5.0000e-04
Epoch 2/30
4453/4454 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step - grid_accuracy: 0.9913 - loss: 0.1210
Epoch 2: val_loss improved from 0.12843 to 0.12229, saving model to eeai_fomo_deep_best.keras

Epoch 2: finished saving model to eeai_fomo_deep_best.keras
4454/4454 ━━━━━━━━━━━━━━━━━━━━ 220s 49ms/step - grid_accuracy: 0.9911 - loss: 0.1185 - val_grid_accuracy: 0.9916 - val_loss: 0.1223 - learning_rate: 5.0000e-04
Epoch 3/30
4453/4454 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step - grid_accuracy: 0.9904 - loss: 0.1104
Epoch 3: val_loss improved from 0.12229 to 0.12174,

In [7]:
# ==============================================================================
# MODEL SUMMARY PER EMBEDDED
# ==============================================================================

def embedded_summary(model, input_shape=(1, 120, 18)):
    
    # 2. Calcola i parametri statici (Flash)
    total_params = model.count_params()
    estimated_flash_kb = (total_params * 4) / 1024
    
   # 3. Calcola il picco di memoria dinamica (SRAM/Tensor Arena)
    max_layer_ram_kb = 0
    for layer in model.layers:
        # AGGIUNTO: Salta l'InputLayer o i layer senza output_shape per evitare l'AttributeError
        if layer.__class__.__name__ == 'InputLayer' or not hasattr(layer, 'output_shape'):
            continue
            
        output_shape = layer.output_shape
        if isinstance(output_shape, list):
            num_elements = sum([np.prod([dim for dim in shape[1:] if dim is not None]) for shape in output_shape])
        else:
            num_elements = np.prod([dim for dim in output_shape[1:] if dim is not None])
            
        layer_ram_kb = (num_elements * 4) / 1024
        if layer_ram_kb > max_layer_ram_kb:
            max_layer_ram_kb = layer_ram_kb

    input_elements = np.prod(input_shape)
    input_ram_kb = (input_elements * 4) / 1024
    peak_arena_kb = input_ram_kb + max_layer_ram_kb

    # 4. Stampa il verdetto 
    print("============================================")
    print("   REPORT REQUISITI HARDWARE (STIMA FLOAT32)   ")
    print("============================================")
    print(f" Memoria FLASH stimata : {estimated_flash_kb:.2f} KB  (Limite : < 800 KB)")
    print(f" Memoria SRAM stimata  : ~{peak_arena_kb:.2f} KB (Limite : < 400 KB)")
    print(" Operazioni Ricorrenti : ASSENTI (RNN/LSTM/GRU non rilevate)")
    print(" Nota sulla Quantizz.  : Raccomandata INT8 per ESP32-S3 (ridurrà la RAM di ~4x)")
    print("============================================\n")

embedded_summary(model_fomo)
#model_heavy.summary()

   REPORT REQUISITI HARDWARE (STIMA FLOAT32)   
 Memoria FLASH stimata : 1972.88 KB  (Limite : < 800 KB)
 Memoria SRAM stimata  : ~8.44 KB (Limite : < 400 KB)
 Operazioni Ricorrenti : ASSENTI (RNN/LSTM/GRU non rilevate)
 Nota sulla Quantizz.  : Raccomandata INT8 per ESP32-S3 (ridurrà la RAM di ~4x)



In [8]:
# ==============================================================================
# VISUALIZZATORE FOMO (Estrae coordinate dalla griglia e disegna l'heatmap)
# ==============================================================================
import scipy.ndimage as ndimage

file_target = "dataset/window_000005.npz" # <-- Controlla il tuo path
if not os.path.exists(file_target):
    print(f"ERRORE: Non trovo il file {file_target}")
else:
    data = np.load(file_target)
    raw_iq = data['radar_cir_iq'] 
    gt_coords = data['people_xy'] 
    gt_mask = data['people_mask'] 
    T = raw_iq.shape[0]

    print("Elaborazione filtri in corso...")
    mag = np.sqrt(raw_iq[..., 0]**2 + raw_iq[..., 1]**2).reshape(T, 1, 120, 18)
    decluttered = np.zeros_like(mag)
    bg = np.copy(mag[0])
    alpha = 0.20
    for t in range(T):
        bg = alpha * mag[t] + (1 - alpha) * bg
        decluttered[t] = np.abs(mag[t] - bg)

    print("Caricamento modello FOMO...")
    model_fomo = load_model(
        "eeai_fomo_best.keras",
        custom_objects={
            "heatmap_focal_loss": heatmap_focal_loss,
            "grid_accuracy": grid_accuracy
        }
    )

    preds = model_fomo.predict(decluttered, verbose=0) # Output shape: (T, 40, 24, 1)
    
    out = widgets.Output() 

    def draw_frame(frame_idx, soglia):
        with out:
            clear_output(wait=True)
            fig, ax = plt.subplots(figsize=(4.8, 7.2))
            ax.set_xlim(0, 4.8); ax.set_ylim(0, 7.2)
            
            # 1. Disegna l'Heatmap predetta come sfondo!
            heatmap = preds[frame_idx, :, :, 0]
            ax.imshow(heatmap, origin='lower', extent=[0, 4.8, 0, 7.2], cmap='magma', alpha=0.7)
            
            ax.set_title(f"Radar FOMO | Frame: {frame_idx}/{T-1}", fontsize=14, fontweight='bold')

            # 2. Disegna la Ground Truth
            for i in range(4):
                if gt_mask[frame_idx, i] > 0.5:
                    rx, ry = gt_coords[frame_idx, i]
                    ax.scatter(rx, ry, c='limegreen', s=120, edgecolors='white', marker='o')
                    ax.text(rx, ry + 0.15, f"GT {i+1}", color='white', fontweight='bold', ha='center')

            # 3. Estrai le coordinate predette (Post-processing stile FOMO)
            # Cerchiamo i punti nella griglia che superano la soglia
            labeled, num_features = ndimage.label(heatmap > soglia)
            for i in range(1, num_features + 1):
                # Trova il centro di massa per ogni blob rilevato
                row_idx, col_idx = ndimage.center_of_mass(heatmap, labeled, i)
                
                # Converti gli indici della griglia in Metri
                px = (col_idx + 0.5) * (4.8 / 24)
                py = (row_idx + 0.5) * (7.2 / 40)
                
                ax.scatter(px, py, c='cyan', s=100, marker='X', edgecolors='black')
                ax.text(px, py - 0.2, "PRED", color='cyan', fontsize=10, ha='center', fontweight='bold')

            plt.xlabel("X (Metri)"); plt.ylabel("Y (Metri)")
            plt.tight_layout(); plt.show()

    slider_frame = widgets.IntSlider(value=500, min=10, max=T-1, step=1, description='Frame:')
    slider_soglia = widgets.FloatSlider(value=0.50, min=0.1, max=0.99, step=0.05, description='Soglia:')

    def on_change(change):
        draw_frame(slider_frame.value, slider_soglia.value)

    slider_frame.observe(on_change, names='value')
    slider_soglia.observe(on_change, names='value')

    controls = widgets.VBox([slider_frame, slider_soglia])
    controls.layout.margin = '20px 20px 20px 0px' 
    ui = widgets.HBox([controls, out])
    
    display(ui)
    draw_frame(slider_frame.value, slider_soglia.value)

Elaborazione filtri in corso...
Caricamento modello FOMO...
